# Analyzing some tendencies after slim dimensionality reduction with Random Embeddings

In this script we just focus on some partial dataset wherein we use $d=20$ on just a subset of functions from BBOB (i.e. $f=\{1,8,11,16,20 \}$). What we want to do is then check how some features from the Exploratory Landscape Analysis Toolbox (i.e. FLACCO) behave under a projection with Random Embeddings with a projection matrix $P \in \mathbb{R}^{d \times q}$. In this case we set $q=10$ as a preliminary value to have $50\%$ reduction.

## Procedure
For this case, we use the package `sklearn.random_projection.GaussianRandomProjection` in order to generate the random projection matrices. This generates random matrices where $\left\|P \right\|_{2}=1.0$. With this in mind, we generate 10 different random projection matrices of this kind. Furthermore, we generated beforehand initial 10 datasets of $\mathbf{X} \in [-5,5]^{d}$ with $100 d$ samples via `LHS`. *A priori*, the sampling scheme wouldn't signify a very different performance with respect to other sampling available techniques since the Random Projection will transform any initial space-filling optimized Design of Experiments scheme into some degenerate one since there'll be artificial correlations caused by the projection into the reduced latent space.

### Sampled ELA Features
After what was discussed in some references (i.e. *Expressiveness and Robustness of Landscape Features*, Renau et al., 2019), we sampled the following sets of ELA Features:
- **Dispersion (disp)**
- **Information Content (ic)**
- **Nearest Best Clustering (nbc)**
- **Principal Component Analysis (pca)**
- **Meta Model (ela_meta)**
- **$y$-distribution (ela_distr)**
- **Level Set (ela_level)**

From the set of features, it's self-evident that the **$y$-distribution** features, these are invariant to any dimensionality reduction techniques since the scope of these features is just the *objective space*. However to account for subsampling in a high dimensional space we proceed to generate subsets of $100 q$ by sampling without replacement from the initial dataset. The idea is then two-fold: 1) We simulate what would happen what happens by keeping the same *dimensionality propertion* and 2) we can study the stability and possible distributions of the features. We hypothesize that the distributions of features by performing this sampling will depend primarily on the sampled function.

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# Import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Define file paths
ela_features_reduced_path = Path("ela_features_reduced/ELA_extraction/D_20").resolve()
ela_features_complete_path = Path("ela_features_2/ELA_extraction/Dimension_20").resolve()

In [3]:
# Get list of CSV files in both directories
reduced_files = sorted(ela_features_reduced_path.rglob("*.csv"))
complete_files = sorted(ela_features_complete_path.rglob("*.csv"))

In [4]:
complete_files

[WindowsPath('C:/Users/iolar/OneDrive - Universiteit Leiden/Documents/dimensionality_reduction repo/ela_dim_red/ela_features_2/ELA_extraction/Dimension_20/seed_123/Samples_2000/f_1/id_0/ela_features.csv'),
 WindowsPath('C:/Users/iolar/OneDrive - Universiteit Leiden/Documents/dimensionality_reduction repo/ela_dim_red/ela_features_2/ELA_extraction/Dimension_20/seed_123/Samples_2000/f_1/id_1/ela_features.csv'),
 WindowsPath('C:/Users/iolar/OneDrive - Universiteit Leiden/Documents/dimensionality_reduction repo/ela_dim_red/ela_features_2/ELA_extraction/Dimension_20/seed_123/Samples_2000/f_1/id_10/ela_features.csv'),
 WindowsPath('C:/Users/iolar/OneDrive - Universiteit Leiden/Documents/dimensionality_reduction repo/ela_dim_red/ela_features_2/ELA_extraction/Dimension_20/seed_123/Samples_2000/f_1/id_11/ela_features.csv'),
 WindowsPath('C:/Users/iolar/OneDrive - Universiteit Leiden/Documents/dimensionality_reduction repo/ela_dim_red/ela_features_2/ELA_extraction/Dimension_20/seed_123/Samples_20

In [5]:
def extract_meta_data_from_reduced_feature_file_path(file_path: Path) -> dict:
    """
    Extracts meta data from the reduced feature file path.
    
    Parameters:
    file_path (Path): Path object of the reduced feature file.
    
    Returns:
    dict: A dictionary containing the extracted meta data.
    """
    """
    Extract key-value numeric metadata from path segments like key_value.
    """

    if not isinstance(file_path, (str, Path)):
        raise ValueError("path must be a string or Path object")

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"The path {file_path} does not exist.")

    metadata = {}

    round_txt = file_path.parts[-1]
    round_number = int(round_txt.split("_")[-1].replace(".csv", ""))
    metadata["round"] = round_number

    embedding_seed_txt = file_path.parts[-2]
    embedding_seed = int(embedding_seed_txt.split("_")[-1])
    metadata["embedding_seed"] = embedding_seed

    reduction_ratio_txt = file_path.parts[-4]
    reduction_ratio = float(reduction_ratio_txt.split("_")[-1])
    metadata["reduction_ratio"] = reduction_ratio

    instance_idx_txt = file_path.parts[-5]
    instance_idx = int(instance_idx_txt.split("_")[-1])
    metadata["instance_idx"] = instance_idx

    function_idx_txt = file_path.parts[-6]
    function_idx = int(function_idx_txt.split("_")[-1])
    metadata["function_idx"] = function_idx

    n_samples_txt = file_path.parts[-7]
    n_samples = int(n_samples_txt.split("_")[-1])
    metadata["n_samples"] = n_samples

    seed_txt = file_path.parts[-8]
    seed = int(seed_txt.split("_")[-1])
    metadata["seed_lhs"] = seed

    dimension_txt = file_path.parts[-9]
    dimension = int(dimension_txt.split("_")[-1])
    metadata["dimension"] = dimension
    
    return metadata

In [6]:
def extract_meta_data_from_complete_feature_file_path(file_path: Path) -> dict:
    """
    Extracts meta data from the complete feature file path.
    
    Parameters:
    file_path (Path): Path object of the complete feature file.
    
    Returns:
    dict: A dictionary containing the extracted meta data.
    """
    """
    Extract key-value numeric metadata from path segments like key_value.
    """

    if not isinstance(file_path, (str, Path)):
        raise ValueError("path must be a string or Path object")

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"The path {file_path} does not exist.")

    metadata = {}



    instance_idx_txt = file_path.parts[-2]
    instance_idx = int(instance_idx_txt.split("_")[-1])
    metadata["instance_idx"] = instance_idx

    function_idx_txt = file_path.parts[-3]
    function_idx = int(function_idx_txt.split("_")[-1])
    metadata["function_idx"] = function_idx

    n_samples_txt = file_path.parts[-4]
    n_samples = int(n_samples_txt.split("_")[-1])
    metadata["n_samples"] = n_samples

    seed_txt = file_path.parts[-5]
    seed = int(seed_txt.split("_")[-1])
    metadata["seed_lhs"] = seed

    dimension_txt = file_path.parts[-6]
    dimension = int(dimension_txt.split("_")[-1])
    metadata["dimension"] = dimension
    
    return metadata

In [8]:
# Generate DataFrames by loading CSV files and combining with metadata
complete_dfs = []

for file_path in complete_files:
    df = pd.read_csv(file_path)
    # Extract metadata for the current file
    meta = extract_meta_data_from_complete_feature_file_path(file_path)
    # Add metadata as new columns to the DataFrame
    for key, value in meta.items():
        df[key] = value
    
    complete_dfs.append(df)

complete_df = (
    pd.concat(complete_dfs, ignore_index=True)
    if complete_dfs
    else pd.DataFrame()
)

In [9]:
complete_df

,ela_meta.lin_simple.adj_r2,ela_meta.lin_simple.intercept,ela_meta.lin_simple.coef.min,ela_meta.lin_simple.coef.max,ela_meta.lin_simple.coef.max_by_min,ela_meta.lin_w_interact.adj_r2,ela_meta.quad_simple.adj_r2,ela_meta.quad_simple.cond,ela_meta.quad_w_interact.adj_r2,ela_meta.costs_runtime,...,fitness_distance.distance_mean,fitness_distance.distance_std,fitness_distance.fitness_mean,fitness_distance.fitness_std,fitness_distance.costs_runtime,instance_idx,function_idx,n_samples,seed_lhs,dimension
0,0.783415,160.866202,0.304231,6.910379,22.714241,0.776188,1.000000,1.000000,1.000000,3.719,...,14.032960,1.887697,58.859327,21.820572,0.015,0,1,2000,123,20
1,0.790511,335.919412,0.094152,7.892473,83.827071,0.783521,1.000000,1.000000,1.000000,3.422,...,14.351657,1.945978,233.261314,19.001955,0.000,1,1,2000,123,20
2,0.803591,270.793181,0.251632,7.823506,31.091061,0.797038,1.000000,1.000000,1.000000,3.031,...,13.927840,1.859788,164.643075,21.439205,0.000,10,1,2000,123,20
3,0.843960,517.166290,0.219431,7.591202,34.594919,0.838754,1.000000,1.000000,1.000000,2.890,...,14.825050,2.064448,395.044391,25.092178,0.000,11,1,2000,123,20
4,0.806906,687.009508,0.287814,7.854378,27.289751,0.800463,1.000000,1.000000,1.000000,2.953,...,14.271534,1.986882,578.350601,19.216727,0.000,12,1,2000,123,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3595,0.046794,398366.995801,643.524763,6904.351120,10.728959,0.110644,0.479422,1.476567,0.579389,3.282,...,13.669415,1.855700,150043.046968,31099.152534,0.000,5,9,2000,132,20
3596,0.030062,394205.575977,44.395970,5585.930410,125.820664,0.091313,0.470815,1.791405,0.601925,3.969,...,13.611165,1.776596,152344.564933,30025.249845,0.000,6,9,2000,132,20
3597,0.047229,395039.718828,382.408780,8648.555393,22.615996,0.119119,0.477345,1.424640,0.583361,3.547,...,14.367946,2.105526,150410.749166,34466.209774,0.000,7,9,2000,132,20
3598,0.043838,400072.201886,53.282595,8651.571769,162.371442,0.089749,0.469374,2.183223,0.566189,3.671,...,13.671871,1.800489,151853.393155,31118.184509,0.000,8,9,2000,132,20


In [17]:
FEATURE_DTYPES = {
    col: "float64"
    for col in pd.read_csv(
        reduced_files[0],
        nrows=0
    ).columns
}

META_DTYPES = {
    "dimension": "int16",
    "seed_lhs": "int32",
    "n_samples": "int32",
    "function_idx": "int16",
    "instance_idx": "int32",
    "reduction_ratio": "float32",
    "embedding_seed": "int32",
    "round": "int16",
}

In [11]:
# Generate DataFrames by loading CSV files and combining with metadata
reduced_dfs = []

for file_path in reduced_files:
    df = pd.read_csv(file_path)
    # Extract metadata for the current file
    meta = extract_meta_data_from_reduced_feature_file_path(file_path)
    # Add metadata as new columns to the DataFrame
    for key, value in meta.items():
        df[key] = value
    
    reduced_dfs.append(df)

reduced_df = (
    pd.concat(reduced_dfs, ignore_index=True)
    if reduced_dfs
    else pd.DataFrame()
)

KeyboardInterrupt: 

In [18]:
len(reduced_files)

247500

In [19]:
def load_reduced(fp):
    # Fast CSV read, no inference
    df = pd.read_csv(
        fp,
        dtype=FEATURE_DTYPES,
        low_memory=False,
        engine="c"
    )

    meta = extract_meta_data_from_reduced_feature_file_path(fp)

    # Typed metadata injection (no object columns)
    return df.assign(
        **{
            k: pd.Series(v, dtype=META_DTYPES[k], index=df.index)
            for k, v in meta.items()
        }
    )



In [20]:
# 3. Concatenate everything
reduced_df = pd.concat(
    map(load_reduced, reduced_files),
    ignore_index=True
)

In [21]:
# Store the final DataFrame
reduced_df.to_parquet("reduced_data.parquet", engine="pyarrow", compression="snappy")


In [22]:
complete_df.to_csv("complete_data.csv", index=False)

In [23]:
reduced_df.to_csv("reduced_data.csv", index=False)